In [1]:

import sys
import types

# --- STEP 1: Fool Pickle by creating a fake 'features' module in memory ---
fake_module = types.ModuleType("features")
sys.modules["features"] = fake_module


# Tell Python how to unpack the custom object into a dummy class
class DummyQueryFeatures:

    def __setstate__(self, state):
        self.__dict__.update(state)


fake_module.QueryFeatures = DummyQueryFeatures

# --- STEP 2: Now, load your pickle file ---
import pandas as pd

saved_data = pd.read_pickle("F:/code_experiment/models/pa_query_data.pkl")

# Extract your variables from the saved dict
feature_rows = saved_data["feature_rows"]
labels = saved_data["labels"]
meta = saved_data["meta"]

# --- STEP 3: Flatten the objects into standard rows/columns ---
flat_rows = []
for feat, label, m in zip(feature_rows, labels, meta):
    # Get the attributes out of our dummy Python object
    row_data = feat.__dict__.copy()

    # Add the target label and metadata keys
    row_data["target_label"] = label
    row_data.update(m)

    flat_rows.append(row_data)

# Create a clean DataFrame (only strings, numbers, booleans)
df_clean = pd.DataFrame(flat_rows)



In [12]:
%pip install jupysql duckdb-engine --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%load_ext sql

In [3]:
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [5]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [6]:
import duckdb

# 1. Create a native DuckDB connection
conn = duckdb.connect()

# 2. Pass this active connection to JupySQL
%sql conn --alias duckdb

In [8]:
%%sql
SELECT has_relevant_index, count(*) as count 
FROM df_clean
group by has_relevant_index
;

,has_relevant_index,count
0,False,25
1,True,6


In [9]:
%%sql
SELECT *
FROM df_clean
;

,node_type,estimated_cost,estimated_rows,plan_width,tables,table_count,filtered_seq_scan_count,index_scan_count,has_relevant_index,seconds_since_last_analyze,n_mod_since_analyze,n_live_tup,is_cross_join,join_count,target_label,label,actual_rows,misestimated
0,Aggregate,4.224162e+05,6.000000e+00,236.0,[lineitem],1,1,0,False,59.240911,0,10987239,False,0,0,Q01,4.000000e+00,False
1,Limit,2.417405e+06,1.000000e+00,192.0,"[part, region, supplier, nation, partsupp]",5,2,5,False,302.657545,0,10065650,True,7,1,Q02,1.000000e+02,True
2,Limit,7.061181e+05,1.000000e+01,44.0,"[lineitem, orders, customer]",3,3,0,False,353.957377,0,22485719,False,2,0,Q03,1.000000e+01,False
3,Aggregate,3.580059e+05,5.000000e+00,24.0,"[orders, lineitem]",2,0,3,False,376.419898,0,20985954,True,1,0,Q04,5.000000e+00,False
4,Sort,4.194206e+05,2.500000e+01,58.0,"[customer, nation, region, orders, lineitem, s...",6,1,2,False,723.953922,0,22585749,True,5,1,Q05,5.000000e+00,True
5,Aggregate,3.105419e+05,1.000000e+00,32.0,[lineitem],1,1,0,False,728.616644,0,10987239,False,0,0,Q06,1.000000e+00,False
6,Aggregate,4.028719e+05,1.002800e+04,116.0,"[customer, nation, orders, lineitem, supplier]",5,3,1,False,880.844542,0,22585744,True,5,1,Q07,4.000000e+00,True
7,Aggregate,2.214583e+05,2.406000e+03,64.0,"[part, lineitem, orders, customer, nation, reg...",7,2,4,False,923.678334,0,24585764,True,7,1,Q08,2.000000e+00,True
8,Aggregate,5.110945e+05,2.300000e+01,90.0,"[partsupp, part, lineitem, supplier, orders, n...",6,1,4,False,1192.571327,0,31051599,True,5,1,Q09,1.750000e+02,True
9,Limit,5.609227e+05,2.000000e+01,201.0,"[lineitem, orders, customer, nation]",4,1,3,False,1250.381650,0,22485744,True,3,0,Q10,2.000000e+01,False


In [48]:
df_clean.dtypes


node_type                         str
estimated_cost                float64
estimated_rows                float64
plan_width                    float64
tables                         object
table_count                     int64
filtered_seq_scan_count         int64
index_scan_count                int64
has_relevant_index             object
seconds_since_last_analyze    float64
n_mod_since_analyze             int64
n_live_tup                      int64
is_cross_join                    bool
join_count                      int64
target_label                    int64
label                             str
actual_rows                   float64
misestimated                     bool
dtype: object

In [50]:

import sys
import types

# --- STEP 1: Fool Pickle by creating a fake 'features' module in memory ---
fake_module = types.ModuleType("features")
sys.modules["features"] = fake_module


# Tell Python how to unpack the custom object into a dummy class
class DummyQueryFeatures:

    def __setstate__(self, state):
        self.__dict__.update(state)


fake_module.QueryFeatures = DummyQueryFeatures

# --- STEP 2: Now, load your pickle file ---

import pandas as pd

saved_data = pd.read_pickle("F:/code_experiment/models/query_data.pkl")
pa_saved_data = pd.read_pickle("F:/code_experiment/models/pa_query_data.pkl")



# Convert saved_data to DataFrame if it is a dictionary
if isinstance(saved_data, dict):
    saved_data = pd.DataFrame([saved_data])  # Use [saved_data] if it's a single row, or pd.DataFrame(saved_data) if it's a dict of lists

# Convert pa_saved_data to DataFrame if it is a dictionary
if isinstance(pa_saved_data, dict):
    pa_saved_data = pd.DataFrame([pa_saved_data])
    

# union_df = pd.concat([saved_data, pa_saved_data], ignore_index=True)

In [ ]:
# Extract your variables from the saved dict
feature_rows = saved_data["feature_rows"] 
labels = saved_data["labels"] 
meta = saved_data["meta"] 

# --- STEP 3: Flatten the objects into standard rows/columns ---
flat_rows = []
for feat, label, m in zip(feature_rows, labels, meta):
    print(feat, label, m)
    # Get the attributes out of our dummy Python object
    row_data = feat.__dict__.copy()

    # Add the target label and metadata keys
    row_data["target_label"] = label
    row_data.update(m)

    flat_rows.append(row_data)

# Create a clean DataFrame (only strings, numbers, booleans)
df_clean = pd.DataFrame(flat_rows)

<class '__main__.DummyQueryFeatures'> 0 {'label': 'Q01', 'tables': ('lineitem',), 'estimated_rows': 6.0, 'actual_rows': 4.0, 'misestimated': False}
<class '__main__.DummyQueryFeatures'> 1 {'label': 'Q02', 'tables': ('part', 'region', 'nation', 'supplier', 'partsupp'), 'estimated_rows': 1.0, 'actual_rows': 100.0, 'misestimated': True}
<class '__main__.DummyQueryFeatures'> 0 {'label': 'Q03', 'tables': ('lineitem', 'orders', 'customer'), 'estimated_rows': 10.0, 'actual_rows': 10.0, 'misestimated': False}
<class '__main__.DummyQueryFeatures'> 0 {'label': 'Q04', 'tables': ('orders', 'lineitem'), 'estimated_rows': 5.0, 'actual_rows': 5.0, 'misestimated': False}
<class '__main__.DummyQueryFeatures'> 1 {'label': 'Q05', 'tables': ('supplier', 'customer', 'nation', 'region', 'orders', 'lineitem'), 'estimated_rows': 25.0, 'actual_rows': 5.0, 'misestimated': True}
<class '__main__.DummyQueryFeatures'> 0 {'label': 'Q06', 'tables': ('lineitem',), 'estimated_rows': 1.0, 'actual_rows': 1.0, 'misestima

<class 'list'>


In [ ]:
# Extract your variables from the saved dict
feature_rows =  [value for values in saved_data["feature_rows"] + pa_saved_data["feature_rows"] for value in values]
labels = [value for values in saved_data["labels"] + pa_saved_data["labels"] for value in values]
meta =  [value for values in saved_data["meta"] + pa_saved_data["meta"] for value in values]

# --- STEP 3: Flatten the objects into standard rows/columns ---
flat_rows = []
for feat, label, m in zip(feature_rows, labels, meta):
    print(feat, label, m)
    # Get the attributes out of our dummy Python object
    row_data = feat.__dict__.copy()

    # Add the target label and metadata keys
    row_data["target_label"] = label
    row_data.update(m)

    flat_rows.append(row_data)

# Create a clean DataFrame (only strings, numbers, booleans)
df_clean = pd.DataFrame(flat_rows)

<__main__.DummyQueryFeatures object at 0x000001F96CA6FCB0> 0 {'label': 'Q01', 'tables': ('lineitem',), 'estimated_rows': 6.0, 'actual_rows': 4.0, 'misestimated': False}
<__main__.DummyQueryFeatures object at 0x000001F96467A490> 1 {'label': 'Q02', 'tables': ('part', 'region', 'nation', 'supplier', 'partsupp'), 'estimated_rows': 1.0, 'actual_rows': 100.0, 'misestimated': True}
<__main__.DummyQueryFeatures object at 0x000001F96467A850> 0 {'label': 'Q03', 'tables': ('lineitem', 'orders', 'customer'), 'estimated_rows': 10.0, 'actual_rows': 10.0, 'misestimated': False}
<__main__.DummyQueryFeatures object at 0x000001F96CB26D70> 0 {'label': 'Q04', 'tables': ('orders', 'lineitem'), 'estimated_rows': 5.0, 'actual_rows': 5.0, 'misestimated': False}
<__main__.DummyQueryFeatures object at 0x000001F96CB268B0> 1 {'label': 'Q05', 'tables': ('supplier', 'customer', 'nation', 'region', 'orders', 'lineitem'), 'estimated_rows': 25.0, 'actual_rows': 5.0, 'misestimated': True}
<__main__.DummyQueryFeatures o

28
